# EXH Marketing Mix Model — Phase 2, Iteration 2
**Script 1 of 3: Model Training & Artifact Export**

Trains the final Mixed Hierarchical MMM (Iter 2) on weekly EXH data. Saves all posteriors, scalers, metadata, and pre-computed response parameters to `ARTIFACT_PATH` so Scripts 2 and 3 can run independently.

### Artifacts Written
| File | Description |
| --- | --- |
| `trace_iter2_mixed_hier.nc` | ArviZ/NetCDF trace (all posterior samples) |
| `scaler_X.pkl` | StandardScaler fitted on train X |
| `scaler_y.pkl` | StandardScaler fitted on train y |
| `spend_train_max_iter2.pkl` | Per-combo max spend (train period) |
| `response_params.pkl` | Posterior mean parameters, keyed by combo |
| `model_meta.pkl` | Structural metadata (combos, splits, etc.) |
| `model_spec.pkl` | Functional spec (saturation types, scaling) |
| `baseline_2025.pkl` | 2025 annual / weekly spend baseline |
| `weekly_patterns_2025.pkl` | 52-week spend proportions per combo (2025) |
| `df_full.parquet` | Full 165-week dataset with predictions |
| `df_2024_backtest.parquet` | 2024-only subset for backtesting |

In [0]:
%pip install pymc arviz numpyro jax jaxlib pytensor scikit-learn openpyxl --quiet

In [0]:
import os
import pickle
import shutil
import datetime

import numpy as np
import pandas as pd
import pytensor
import pytensor.tensor as pt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import pymc as pm
import arviz as az
import pymc.sampling.jax as pmjax

## 1. Configuration

In [0]:
FILE_PATH     = r"/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/EXH_ModelData.xlsx"
ARTIFACT_PATH = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_Outcomes/"

os.makedirs(ARTIFACT_PATH, exist_ok=True)

print(f"PyMC  : {pm.__version__}")
print(f"ArviZ : {az.__version__}")
print(f"Artifacts → {ARTIFACT_PATH}")

## 2. Data Loading & Feature Engineering
Loads the weekly EXH model data, encodes binary flag columns, creates season dummies (winter as reference), and adds year indicators + linear trend.

In [0]:
df = pd.read_excel(FILE_PATH, sheet_name=0)
df = df.sort_values("week_date").reset_index(drop=True)

# Binary encode flag columns
df["stat_holiday_in_week"] = (df["stat_holiday_in_week"] == "Y").astype(int)
df["is_long_weekend"]      = (df["is_long_weekend"]      == "Y").astype(int)

# Season dummies (winter = reference, dropped)
season_dummies = pd.get_dummies(df["season"], prefix="season", drop_first=False).astype(int)
season_dummies = season_dummies.drop(columns=["season_winter"])
df = pd.concat([df.drop(columns=["season"]), season_dummies], axis=1)

# Year indicators and linear trend
df["is_2024"]    = (df["week_date"].dt.year == 2024).astype(int)
df["is_2025"]    = (df["week_date"].dt.year == 2025).astype(int)
df["week_index"] = (df["week_date"] - df["week_date"].min()).dt.days / 7

print(f"Data loaded — shape {df.shape}")
print(f"Date range: {df['week_date'].min().date()} → {df['week_date'].max().date()}")
print(f"Missing values: {df.isna().sum().sum()}")

## 3. Feature Selection
26 features dropped in Phase 1 due to multicollinearity / low VIF signal. Baseline control columns (cols 3–44) are filtered, augmented with season dummies and trend indicators → 22 features total.

In [0]:
date_col   = "week_date"
target_col = "total_weekly_attendance"

DROP_FEATURES = [
    "national_max_temp", "national_min_temp", "national_temp_anomaly",
    "action_intensity", "adaptation_intensity", "adventure_intensity",
    "animated_intensity", "biopic_intensity", "comedy_intensity",
    "crime_intensity", "documentary_intensity", "drama_intensity",
    "family_intensity", "fantasy_intensity", "foreign_intensity",
    "horror_intensity", "musical_intensity", "mystery_intensity",
    "period_intensity", "political_intensity", "romance_intensity",
    "romanticcomedy_intensity", "scifi_intensity", "sequel_intensity",
    "suspense_intensity", "thriller_intensity",
]

control_cols     = df.columns[3:44].tolist()
control_cols     = [c for c in control_cols if c != "season"] + season_dummies.columns.tolist()
baseline_features = [f for f in control_cols if f not in DROP_FEATURES]
baseline_features += ["is_2024", "is_2025", "week_index"]

assert len(baseline_features) == 22, f"Expected 22 baseline features, got {len(baseline_features)}"
print(f"Baseline features: {len(baseline_features)}")

## 4. Train / Validation / Test Split
Chronological split: last 15 weeks held out as test, preceding 26 weeks as validation.

In [0]:
n       = len(df)
n_test  = 15
n_val   = 26
n_train = n - n_val - n_test

train_df = df.iloc[:n_train].copy().reset_index(drop=True)
val_df   = df.iloc[n_train : n_train + n_val].copy().reset_index(drop=True)
test_df  = df.iloc[n_train + n_val :].copy().reset_index(drop=True)

print(f"Train : {n_train} weeks  ({train_df['week_date'].min().date()} → {train_df['week_date'].max().date()})")
print(f"Val   : {n_val}   weeks  ({val_df['week_date'].min().date()} → {val_df['week_date'].max().date()})")
print(f"Test  : {n_test}  weeks  (held out)")

## 5. Scaling
StandardScalers fitted on train only; validation and test use the same fitted parameters.

In [0]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(train_df[baseline_features])
X_val   = scaler_X.transform(val_df[baseline_features])
# X_test  = scaler_X.transform(test_df[baseline_features])

y_train = scaler_y.fit_transform(train_df[[target_col]]).flatten()
y_val   = scaler_y.transform(val_df[[target_col]]).flatten()

## 6. Media Metadata & Funnel×LOB Aggregation
Parses raw media columns into funnel/LOB/channel structure, groups sparse LOBs into interpretable business groups, and builds the 10-term collapsed combo structure. Three sparse combos (>60% zeros in train) are dropped.

In [0]:
all_cols    = df.columns.tolist()
target_idx  = all_cols.index(target_col)
media_start = all_cols.index("Awareness_Cineclub_Digital")
media_cols  = all_cols[media_start:target_idx]

def parse_media_col(col):
    parts   = col.split("_")
    funnel  = parts[0].capitalize()
    channel = parts[-1]
    lob     = "_".join(parts[1:-1])
    return funnel, lob, channel

media_meta = pd.DataFrame(
    [parse_media_col(c) + (c,) for c in media_cols],
    columns=["funnel", "lob", "channel", "col_name"],
)

print(f"Media columns   : {len(media_cols)}")
print("Funnels         :", sorted(media_meta["funnel"].unique()))
print("LOBs            :", sorted(media_meta["lob"].unique()))

# LOB grouping — collapses sparse LOBs into interpretable groups
LOB_GROUPS = {
    "Premium_Brand" : [
        "Premium Experiences - AVX",
        "Premium Experiences - VIP",
        "F&B",
        "EXH - Brand",
    ],
    "Film_Marketing": [
        "Premium Experiences - IMAX",
        "Film Marketing - Inform",
        "Performance - Convert & Upgrade",
    ],
    "Prepaid"       : ["Prepaid - B2B", "Prepaid - B2C"],
    "Cineclub"      : ["Cineclub"],
    "GE"            : ["G&E"],
}

FUNNELS = ["Awareness", "Consideration", "Conversion"]

# Build funnel x LOB group columns on df
for funnel in FUNNELS:
    for lob_group, lobs in LOB_GROUPS.items():
        col_name = f"{funnel}__{lob_group}"
        matching = media_meta.loc[
            (media_meta["funnel"] == funnel) & (media_meta["lob"].isin(lobs)),
            "col_name",
        ].tolist()
        df[col_name] = df[matching].sum(axis=1) if matching else 0.0

In [0]:
# ── PRE-COMBO SPARSITY AUDIT ──────────────────────────────────────────────────
# Shows sparsity of each individual Funnel × LOB BEFORE any grouping or combo
# aggregation. Uses the original LOBs from media_meta.

train_slice = df.iloc[:n_train]
ALL_LOBS = sorted(media_meta["lob"].unique())

rows = []
for funnel in FUNNELS:
    for lob in ALL_LOBS:
        raw_cols = media_meta.loc[
            (media_meta["funnel"] == funnel) & (media_meta["lob"] == lob), "col_name"
        ].tolist()
        if not raw_cols:
            continue
        vals = train_slice[raw_cols].sum(axis=1).values.astype(float)
        n_zero = int((vals == 0).sum())
        pct_zero = n_zero / len(vals) * 100
        max_val = vals.max()
        non_zero_wks = len(vals) - n_zero
        mean_nonzero = vals[vals > 0].mean() if (vals > 0).any() else 0
        total_spend = vals.sum()
        status = "⚠️ SPARSE" if pct_zero > 60 else "Active"
        rows.append({
            "Funnel": funnel,
            "LOB": lob,
            "Total Spend ($)": round(total_spend, 0),
            "Train Max ($)": round(max_val, 0),
            "Zero %": round(pct_zero, 1),
            "Non-zero Wks": f"{non_zero_wks} / {len(vals)}",
            "Mean Non-zero ($)": round(mean_nonzero, 0),
            "Status": status,
        })

sparsity_df = pd.DataFrame(rows)
print(f"Train period: {len(train_slice)} weeks | Threshold for dropping: >60% zero weeks\n")
display(sparsity_df)

In [0]:
# 10-term collapsed combo structure
# Three sparse combos are dropped after zero-week audit (>60% zeros in train)
ITER2_COMBOS = {
    "Awareness__Premium_Brand"     : ["Awareness__Premium_Brand"],
    "Awareness__Other"             : ["Awareness__Cineclub", "Awareness__Film_Marketing"],
    "Consideration__Premium_Brand" : ["Consideration__Premium_Brand"],
    "Consideration__GE"            : ["Consideration__GE"],
    "Consideration__Prepaid"       : ["Consideration__Prepaid"],
    "Consideration__Other"         : ["Consideration__Film_Marketing", "Consideration__Cineclub"],
    "Conversion__Film_Marketing"   : ["Conversion__Film_Marketing"],
    "Conversion__Premium_Brand"    : ["Conversion__Premium_Brand"],
    "Conversion__Cineclub"         : ["Conversion__Cineclub"],
    "Conversion__Other"            : ["Conversion__Prepaid", "Conversion__GE"],
}

DROP_SPARSE = [
    "Awareness__Other",              # 79.8% zero weeks, $3,758 max
    "Consideration__Other",          # 70.2% zero weeks — causes divergences (17) & low ESS (224)
    "Conversion__Other",             # 94.4% zero weeks, $7,556 max
]

# Aggregate source columns into combo columns
for combo_name, source_cols in ITER2_COMBOS.items():
    df[combo_name] = df[source_cols].sum(axis=1)

# Propagate to splits
for split in [train_df, val_df, test_df]:
    for combo_name in ITER2_COMBOS:
        split[combo_name] = df.loc[
            df["week_date"].isin(split["week_date"]), combo_name
        ].values

ACTIVE_COMBOS = [c for c in ITER2_COMBOS if c not in DROP_SPARSE]

FUNNEL_COMBOS = {
    funnel: [c for c in ACTIVE_COMBOS if c.startswith(funnel)]
    for funnel in FUNNELS
}

LOB_GROUP_MAP = {lob_group: lobs for lob_group, lobs in LOB_GROUPS.items()}

print(f"Active combos: {len(ACTIVE_COMBOS)}")
for funnel, combos in FUNNEL_COMBOS.items():
    print(f"  {funnel:<15}: {combos}")

## 7. Media Spend Scaling
Each combo is max-normalised on the train period so spend ∈ [0, 1] during training. The same max is used at inference time to ensure consistency.

In [0]:
spend_scaled_iter2    = {}
spend_train_max_iter2 = {}

print(f"{'Combination':<35} {'Train max':>14}  Scaled range")
print("─" * 70)
for combo_name in ACTIVE_COMBOS:
    m          = float(train_df[combo_name].max())
    train_vals = train_df[combo_name].values.astype(np.float64)
    val_vals   = val_df[combo_name].values.astype(np.float64)
    trainval   = np.concatenate([train_vals, val_vals])

    spend_train_max_iter2[combo_name] = m
    spend_scaled_iter2[combo_name]    = {
        "train"    : train_vals / m,
        "trainval" : trainval   / m,
    }
    print(f"{combo_name:<35} ${m:>12,.0f}  "
          f"[{train_vals.min()/m:.3f}, {train_vals.max()/m:.3f}]")

In [0]:
# ── SPARSITY AUDIT ──
print(f"{'Combo':<35} {'Train Max':>10} {'Zero %':>8} {'Non-zero wks':>13} {'Status'}")
print("─" * 85)
for combo_name in ITER2_COMBOS:
    vals = train_df[combo_name].values.astype(float)
    n_zero = int((vals == 0).sum())
    pct_zero = n_zero / len(vals) * 100
    max_val = vals.max()
    status = "DROPPED" if combo_name in DROP_SPARSE else "ACTIVE ✓"
    print(f"{combo_name:<35} ${max_val:>9,.0f} {pct_zero:>7.1f}% {len(vals) - n_zero:>8} / {len(vals)}   {status}")

print(f"\n{'─' * 85}")
print(f"Train period: {len(train_df)} weeks")
print(f"Active combos: {len(ACTIVE_COMBOS)}")
print(f"Dropped combos: {len(DROP_SPARSE)}")

# Underlying raw columns per active combo
print(f"\n\n{'═' * 85}")
print("UNDERLYING RAW MEDIA COLUMNS PER ACTIVE COMBO")
print(f"{'═' * 85}\n")

for combo in ACTIVE_COMBOS:
    source_cols = ITER2_COMBOS[combo]
    print(f"▸ {combo}")
    for src in source_cols:
        parts = src.split("__")
        funnel_name = parts[0]
        lob_group_name = parts[1]
        if lob_group_name in LOB_GROUP_MAP:
            lobs = LOB_GROUP_MAP[lob_group_name]
            for lob in lobs:
                raw = media_meta.loc[
                    (media_meta["funnel"] == funnel_name) & (media_meta["lob"] == lob), "col_name"
                ].tolist()
                if raw:
                    total_spend = train_df[raw].sum().sum()
                    zero_pct = ((train_df[raw].sum(axis=1) == 0).sum() / len(train_df)) * 100
                    print(f"    {lob:<40} {len(raw)} cols  ${total_spend:>10,.0f}  {zero_pct:.0f}% zero weeks")
                else:
                    print(f"    {lob:<40} ⚠️  NO RAW COLUMNS at {funnel_name} level")
    print()

## 8. Response Transformation Functions (PyTensor)
Three transformations used inside the PyMC model graph:
* **Geometric adstock** — `adstocked[t] = x[t] + α · adstocked[t-1]`
* **Hill saturation** (Consideration) — `x^α_s / (x^α_s + k^α_s)`
* **Logistic saturation** (Awareness/Conversion) — `1 - exp(-λ · x)`

In [0]:
def geometric_adstock(x, alpha):
    """
    Recursive geometric adstock: adstocked[t] = x[t] + alpha * adstocked[t-1]
    """
    x_t = pt.as_tensor_variable(x.astype(np.float64))

    def step(x_curr, adstock_prev, alpha):
        return x_curr + alpha * adstock_prev

    adstocked, _ = pytensor.scan(
        fn=step,
        sequences=[x_t],
        outputs_info=[pt.as_tensor_variable(np.float64(0.0))],
        non_sequences=[alpha],
    )
    return adstocked


def hill_saturation(x, k, alpha_s):
    """Hill saturation: f(x) = x^alpha_s / (x^alpha_s + k^alpha_s)"""
    x_safe = pt.maximum(x, 1e-6)
    return (x_safe ** alpha_s) / (x_safe ** alpha_s + k ** alpha_s)


def logistic_saturation(x, lam):
    """Logistic saturation: f(x) = 1 - exp(-lam * x)"""
    return 1 - pt.exp(-lam * x)

## 9. Model Definition — Iteration 2 Mixed Hierarchical

**Hierarchy:**
* Funnel-level hyperpriors pool adstock (α) and effect size (γ) across all LOB combos within the funnel
* **Consideration**: Hill saturation + pooled `k` and `α_s` hyperpriors
* **Awareness / Conversion**: Logistic saturation, independent `λ` per combo

**Prior design:**
* α priors favour high carryover for Awareness (mean ≈ 0.78), low for Conversion (mean ≈ 0.29)
* `μ_γ` priors reflect expected contribution magnitude per funnel

In [0]:
ALPHA_MU_PRIOR = {
    "Awareness"    : {"alpha": 7, "beta": 2},   # mean ≈ 0.78 — long carryover
    "Consideration": {"alpha": 3, "beta": 3},   # mean ≈ 0.50 — moderate
    "Conversion"   : {"alpha": 2, "beta": 5},   # mean ≈ 0.29 — short carryover
}

MU_GAMMA_PRIOR = {
    "Awareness"    : 0.5,
    "Consideration": 0.3,
    "Conversion"   : 0.6,
}

with pm.Model() as mmm_iter2_mixed_hier:

    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_train.shape[1])
    sigma     = pm.HalfNormal("sigma", sigma=1)

    mu = intercept + pm.math.dot(X_train, beta)

    for funnel, combos in FUNNEL_COMBOS.items():
        ap = ALPHA_MU_PRIOR[funnel]

        # Funnel-level hyperpriors
        mu_alpha_f    = pm.Beta(f"mu_alpha_{funnel}", alpha=ap["alpha"], beta=ap["beta"])
        kappa_alpha_f = pm.HalfNormal(f"kappa_alpha_{funnel}", sigma=2)
        mu_gamma_f    = pm.HalfNormal(f"mu_gamma_{funnel}", sigma=MU_GAMMA_PRIOR[funnel])

        if funnel == "Consideration":
            mu_k_f       = pm.HalfNormal("mu_k_Consideration",       sigma=0.5)
            mu_alpha_s_f = pm.HalfNormal("mu_alpha_s_Consideration", sigma=1.5)

        for combo in combos:
            # Adstock decay — pooled via Beta reparameterisation
            alpha_c = pm.Beta(
                f"alpha_{combo}",
                alpha = mu_alpha_f    * kappa_alpha_f,
                beta  = (1 - mu_alpha_f) * kappa_alpha_f,
            )

            # Effect size — pooled via half-normal hyperprior
            gamma_c = pm.HalfNormal(f"gamma_{combo}", sigma=mu_gamma_f)

            adstocked = geometric_adstock(spend_scaled_iter2[combo]["train"], alpha_c)

            if funnel == "Consideration":
                k_c       = pm.HalfNormal(f"k_{combo}",       sigma=mu_k_f)
                alpha_s_c = pm.HalfNormal(f"alpha_s_{combo}", sigma=mu_alpha_s_f)
                saturated = hill_saturation(adstocked, k_c, alpha_s_c)
            else:
                lam_c     = pm.HalfNormal(f"lam_{combo}", sigma=1)
                saturated = logistic_saturation(adstocked, lam_c)

            mu = mu + gamma_c * saturated

    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

    print("Sampling — Iter 2 Mixed Hierarchical (NUTS via NumPyro)...")
    trace_iter2_mixed_hier = pmjax.sample_numpyro_nuts(
        draws=2000,
        tune=2000,
        target_accept=0.99,
        chains=4,
        random_seed=42,
    )

## 10. Convergence Diagnostics
Checks R-hat (< 1.01), effective sample size (> 400), and divergences (target 0).

In [0]:
HYPER_PARAMS = (
    [f"mu_alpha_{f}"    for f in FUNNEL_COMBOS] +
    [f"kappa_alpha_{f}" for f in FUNNEL_COMBOS] +
    [f"mu_gamma_{f}"    for f in FUNNEL_COMBOS] +
    ["mu_k_Consideration", "mu_alpha_s_Consideration"]
)

LOB_PARAMS = []
for funnel, combos in FUNNEL_COMBOS.items():
    LOB_PARAMS += [f"alpha_{c}" for c in combos]
    LOB_PARAMS += [f"gamma_{c}" for c in combos]
    if funnel == "Consideration":
        LOB_PARAMS += [f"k_{c}"       for c in combos]
        LOB_PARAMS += [f"alpha_s_{c}" for c in combos]
    else:
        LOB_PARAMS += [f"lam_{c}" for c in combos]

ALL_PARAMS = ["intercept", "beta", "sigma"] + HYPER_PARAMS + LOB_PARAMS

summary = az.summary(trace_iter2_mixed_hier, var_names=ALL_PARAMS)
summary["r_hat"] = pd.to_numeric(summary["r_hat"], errors="coerce")

max_rhat    = float(summary["r_hat"].max())
min_ess     = int(summary["ess_bulk"].min())
divergences = int(trace_iter2_mixed_hier.sample_stats.diverging.sum().item())

print("=" * 55)
print("CONVERGENCE DIAGNOSTICS")
print("=" * 55)
print(f"Max R-hat   : {max_rhat:.4f}   (target < 1.01)")
print(f"Min ESS     : {min_ess}        (target > 400)")
print(f"Divergences : {divergences}    (target 0)")
print("\n--- Hyperparameter posteriors ---")
print(summary.loc[
    summary.index.intersection(HYPER_PARAMS),
    ["mean", "sd", "eti89_lb", "eti89_ub", "ess_bulk", "r_hat"]
].to_string())

## 11. In-Sample Accuracy Metrics
Posterior means are used to build point predictions. These are the primary accuracy metrics tracked across iterations.

In [0]:
post           = trace_iter2_mixed_hier.posterior
intercept_mean = float(post["intercept"].values.ravel().mean())
beta_mean      = post["beta"].values.reshape(-1, len(baseline_features)).mean(axis=0)

X_trainval  = np.vstack([X_train, X_val])
mu_controls = intercept_mean + X_trainval @ beta_mean
mu_media    = np.zeros(n_train + n_val)
media_contrib = {}

for funnel, combos in FUNNEL_COMBOS.items():
    for combo in combos:
        alpha_mean = float(post[f"alpha_{combo}"].values.ravel().mean())
        gamma_mean = float(post[f"gamma_{combo}"].values.ravel().mean())
        x_spend    = spend_scaled_iter2[combo]["trainval"]

        adstocked    = np.zeros(len(x_spend))
        adstocked[0] = x_spend[0]
        for t in range(1, len(x_spend)):
            adstocked[t] = x_spend[t] + alpha_mean * adstocked[t - 1]

        if funnel == "Consideration":
            k_mean   = float(post[f"k_{combo}"].values.ravel().mean())
            as_mean  = float(post[f"alpha_s_{combo}"].values.ravel().mean())
            x_safe   = np.maximum(adstocked, 1e-6)
            sat      = (x_safe ** as_mean) / (x_safe ** as_mean + k_mean ** as_mean)
        else:
            lam_mean = float(post[f"lam_{combo}"].values.ravel().mean())
            sat      = 1 - np.exp(-lam_mean * adstocked)

        contrib_scaled        = gamma_mean * sat
        mu_media             += contrib_scaled
        media_contrib[combo]  = contrib_scaled * scaler_y.scale_[0]

pred_trainval  = scaler_y.inverse_transform(
                     (mu_controls + mu_media).reshape(-1, 1)).ravel()

pred_train_v   = pred_trainval[:n_train]
pred_val_v     = pred_trainval[n_train:]
actual_train_v = train_df[target_col].values
actual_val_v   = val_df[target_col].values

mape_train = mean_absolute_percentage_error(actual_train_v, pred_train_v) * 100
mape_val   = mean_absolute_percentage_error(actual_val_v,   pred_val_v)   * 100
r2_train   = r2_score(actual_train_v, pred_train_v)
r2_val     = r2_score(actual_val_v,   pred_val_v)

resid_train = actual_train_v - pred_train_v
resid_val   = actual_val_v   - pred_val_v

print("=" * 72)
print("FINAL MODEL ACCURACY — Iter 2 Mixed Hierarchical")
print("=" * 72)
print(f"{'':28} {'Train MAPE':>11} {'Val MAPE':>10} {'Train R²':>10} {'Val R²':>8}")
print("─" * 72)
print(f"{'Mixed Hierarchical Iter 2':28} {mape_train:>10.2f}% {mape_val:>9.2f}% "
      f"{r2_train:>10.3f} {r2_val:>8.3f}  ← FINAL")
print("\n--- Residual summary ---")
print(f"{'':20} {'Train':>15} {'Val':>15}")
print(f"{'Mean':20} {resid_train.mean():>15,.0f} {resid_val.mean():>15,.0f}")
print(f"{'Std':20} {resid_train.std():>15,.0f} {resid_val.std():>15,.0f}")
print(f"{'Median |err|':20} {np.median(np.abs(resid_train)):>15,.0f} "
      f"{np.median(np.abs(resid_val)):>15,.0f}")

## 12. Artifact Export
All downstream scripts load from `ARTIFACT_PATH` — nothing is recomputed. The NetCDF trace is written to local `/tmp` first (HDF5 requires random-access I/O not supported by UC Volume FUSE mounts), then copied to the Volume.

In [0]:
%pip install h5netcdf --quiet

In [0]:
print("=" * 60)
print("SAVING ARTIFACTS")
print("=" * 60)

# 1. Trace (write locally first — HDF5/NetCDF needs random-access I/O)
_local_nc = "/tmp/trace_iter2_mixed_hier.nc"
if os.path.exists(_local_nc):
    os.remove(_local_nc)

_vol_nc = f"{ARTIFACT_PATH}trace_iter2_mixed_hier.nc"
if os.path.exists(_vol_nc):
    os.remove(_vol_nc)

trace_iter2_mixed_hier.to_netcdf(_local_nc)
shutil.copy2(_local_nc, _vol_nc)
os.remove(_local_nc)
print(f"✓ Trace saved → {_vol_nc}")

# 2. Scalers
with open(f"{ARTIFACT_PATH}scaler_X.pkl", "wb") as f:
    pickle.dump(scaler_X, f)
with open(f"{ARTIFACT_PATH}scaler_y.pkl", "wb") as f:
    pickle.dump(scaler_y, f)
print("✓ Scalers saved")

# 3. Spend normalisation constants
with open(f"{ARTIFACT_PATH}spend_train_max_iter2.pkl", "wb") as f:
    pickle.dump(spend_train_max_iter2, f)
print("✓ Spend train maxima saved")

In [0]:
# 4. Structural metadata
model_meta = {
    "ACTIVE_COMBOS"    : ACTIVE_COMBOS,
    "FUNNEL_COMBOS"    : {k: list(v) for k, v in FUNNEL_COMBOS.items()},
    "LOB_GROUP_MAP"    : LOB_GROUP_MAP,
    "baseline_features": baseline_features,
    "target_col"       : target_col,
    "n_train"          : n_train,
    "n_val"            : n_val,
    "n_test"           : n_test,
}
with open(f"{ARTIFACT_PATH}model_meta.pkl", "wb") as f:
    pickle.dump(model_meta, f)
print("✓ Model metadata saved")

# 5. Posterior mean parameters per combo (compact — no full trace required by Script 3)
response_params = {}
for funnel, combos in FUNNEL_COMBOS.items():
    for combo in combos:
        params = {
            "funnel"    : funnel,
            "alpha"     : float(post[f"alpha_{combo}"].values.ravel().mean()),
            "gamma"     : float(post[f"gamma_{combo}"].values.ravel().mean()),
            "saturation": "hill" if funnel == "Consideration" else "logistic",
        }
        if funnel == "Consideration":
            params["k"]       = float(post[f"k_{combo}"].values.ravel().mean())
            params["alpha_s"] = float(post[f"alpha_s_{combo}"].values.ravel().mean())
        else:
            params["lam"] = float(post[f"lam_{combo}"].values.ravel().mean())
        response_params[combo] = params

with open(f"{ARTIFACT_PATH}response_params.pkl", "wb") as f:
    pickle.dump(response_params, f)
print(f"✓ Response parameters saved ({len(response_params)} combos)")

# 6. Model specification (functional signatures for Script 3 optimiser)
model_spec = {
    "adstock_type"    : "geometric",
    "adstock_formula" : "x[t] + alpha * adstocked[t-1]",
    "saturation_types": {
        "Awareness"    : {"type": "logistic", "formula": "1 - exp(-lam * x)"},
        "Consideration": {"type": "hill",     "formula": "x^alpha_s / (x^alpha_s + k^alpha_s)"},
        "Conversion"   : {"type": "logistic", "formula": "1 - exp(-lam * x)"},
    },
    "spend_scaling"  : "max_normalised_on_train",
    "target_scaling" : "standard_scaler",
    "scale_factor"   : float(scaler_y.scale_[0]),
}
with open(f"{ARTIFACT_PATH}model_spec.pkl", "wb") as f:
    pickle.dump(model_spec, f)
print("✓ Model specification saved")

In [0]:
# 7. 2025 annual / weekly spend baseline per combo
df_2025       = df[df["week_date"].dt.year == 2025].copy()
baseline_2025 = {}
for combo in ACTIVE_COMBOS:
    baseline_2025[combo] = {
        "annual_spend": float(df_2025[combo].sum()),
        "weekly_avg"  : float(df_2025[combo].mean()),
        "weeks_active": int((df_2025[combo] > 0).sum()),
    }
with open(f"{ARTIFACT_PATH}baseline_2025.pkl", "wb") as f:
    pickle.dump(baseline_2025, f)
print("✓ 2025 baseline saved")

# 8. 52-week spend proportions per combo (seasonality-aware allocation for Script 3)
weekly_patterns = {}
for combo in ACTIVE_COMBOS:
    combo_spend = df_2025[combo].values
    total       = combo_spend.sum()
    pattern     = (combo_spend / total) if total > 0 else np.ones(len(combo_spend)) / len(combo_spend)
    weekly_patterns[combo] = {
        "proportions"  : pattern.tolist(),
        "weeks_nonzero": int((combo_spend > 0).sum()),
        "peak_week_idx": int(np.argmax(combo_spend)),
    }
with open(f"{ARTIFACT_PATH}weekly_patterns_2025.pkl", "wb") as f:
    pickle.dump(weekly_patterns, f)
print("✓ Weekly spend patterns saved")

In [0]:
# 9. Full 165-week dataset with model predictions (used by Scripts 2 and 3)
df_full = pd.concat([train_df, val_df, test_df]).copy().reset_index(drop=True)
df_full["segment"] = ["train"] * n_train + ["val"] * n_val + ["test"] * n_test
df_full["year"]    = df_full["week_date"].dt.year

X_test           = scaler_X.transform(test_df[baseline_features])
X_full           = np.vstack([X_train, X_val, X_test])
mu_controls_full = intercept_mean + X_full @ beta_mean
mu_media_full    = np.zeros(n_train + n_val + n_test)
media_contrib_full = {}

for funnel, combos in FUNNEL_COMBOS.items():
    for combo in combos:
        alpha_mean = float(post[f"alpha_{combo}"].values.ravel().mean())
        gamma_mean = float(post[f"gamma_{combo}"].values.ravel().mean())
        max_spend  = spend_train_max_iter2[combo]
        x_spend    = df_full[combo].values.astype(np.float64) / max_spend

        adstocked    = np.zeros(len(x_spend))
        adstocked[0] = x_spend[0]
        for t in range(1, len(x_spend)):
            adstocked[t] = x_spend[t] + alpha_mean * adstocked[t - 1]

        if funnel == "Consideration":
            k_mean  = float(post[f"k_{combo}"].values.ravel().mean())
            as_mean = float(post[f"alpha_s_{combo}"].values.ravel().mean())
            x_safe  = np.maximum(adstocked, 1e-6)
            sat     = (x_safe ** as_mean) / (x_safe ** as_mean + k_mean ** as_mean)
        else:
            lam_mean = float(post[f"lam_{combo}"].values.ravel().mean())
            sat      = 1 - np.exp(-lam_mean * adstocked)

        contrib                    = gamma_mean * sat * scaler_y.scale_[0]
        mu_media_full             += gamma_mean * sat
        media_contrib_full[combo]  = contrib

df_full["pred"]          = scaler_y.inverse_transform(
                               (mu_controls_full + mu_media_full).reshape(-1, 1)).ravel()
df_full["controls_pred"] = scaler_y.inverse_transform(
                               mu_controls_full.reshape(-1, 1)).ravel()
df_full["total_media"]   = sum(media_contrib_full[c] for c in ACTIVE_COMBOS)

for combo in ACTIVE_COMBOS:
    df_full[f"media_{combo}"] = media_contrib_full[combo]
for funnel, combos in FUNNEL_COMBOS.items():
    df_full[f"media_{funnel}"] = sum(df_full[f"media_{combo}"] for combo in combos)

df_full.to_parquet(f"{ARTIFACT_PATH}df_full.parquet", index=False)
df[df["week_date"].dt.year == 2024].to_parquet(
    f"{ARTIFACT_PATH}df_2024_backtest.parquet", index=False
)
print("✓ Full dataset parquet saved")

In [0]:
print(f"\n{'='*60}")
print(f"All artifacts → {ARTIFACT_PATH}")
print(f"{'='*60}")
for fname in sorted(os.listdir(ARTIFACT_PATH)):
    fsize = os.path.getsize(f"{ARTIFACT_PATH}{fname}") / 1e6
    print(f"  {fname:<45} {fsize:.1f} MB")